In [1]:
import pandas as pd
import numpy as np
import os

# 1. Load the cleaned intermediate data
input_path = '../data/intermediate/gaia_clean.csv'
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Cleaned dataset not found at {input_path}. Run notebook 03 first.")

df = pd.read_csv(input_path)

# 2. Photometric Feature Engineering: Absolute Magnitude
# Formula: M_G = m_G + 5 * log10(parallax) - 10
df['absolute_magnitude_g'] = df['phot_g_mean_mag'] + 5 * np.log10(df['parallax']) - 10

# 3. Kinematic Feature Engineering: Total Proper Motion Magnitude
# Combines proper motion components in RA and Dec into a single vector magnitude
df['proper_motion_total'] = np.sqrt(df['pmra']**2 + df['pmdec']**2)

# 4. Tangential Velocity Estimation (v_t ≈ 4.74 * pm / parallax)
# Converts proper motion and distance parallax into physical transverse velocity (km/s)
df['tangential_velocity'] = 4.74 * df['proper_motion_total'] / df['parallax']

# 5. Select and structure the final feature matrix for unsupervised ML
feature_columns = [
    'source_id',
    'absolute_magnitude_g',
    'bp_rp',
    'proper_motion_total',
    'tangential_velocity',
    'parallax_snr'
]

feature_matrix = df[feature_columns].dropna().copy()

# 6. Save the processed feature matrix
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'feature_matrix.csv')
feature_matrix.to_csv(output_path, index=False)

print(f"Feature engineering complete. Processed matrix saved to {output_path}")
print(f"Total valid samples: {len(feature_matrix)}")
display(feature_matrix.head())

Feature engineering complete. Processed matrix saved to ../data/processed/feature_matrix.csv
Total valid samples: 24021


,source_id,absolute_magnitude_g,bp_rp,proper_motion_total,tangential_velocity,parallax_snr
0,137342093370833408,10.475110,2.627399,5.264533,5.878481,44.467092
1,137346392632994944,4.423396,0.886630,12.166480,40.679160,80.977058
2,137346564431685120,8.198129,1.967855,31.598544,47.374242,82.136265
3,137349519369183744,5.790543,1.150556,0.947780,4.101884,30.373725
4,137349691167876864,5.374709,1.048895,6.790614,22.656072,56.373397
